In [57]:
import janitor, spacy, recordlinkage, pandas as pd

import numpy as np
from sklearn.preprocessing import StandardScaler
 
import splink as splink

import missingno as msng
from textblob import TextBlob
import great_expectations as ge # great expectations and vibes

#escaleno = StandardScaler() #o cubero?
#alpinsta = escaleno.fit_transform()


df = pd.read_csv("./data.csv") 
df_base = df.copy()
ai_Buffer = df 
df.describe()


,MILES*
count,1155.000000
mean,10.566840
std,21.579106
min,0.500000
25%,2.900000
50%,6.000000
75%,10.400000
max,310.300000


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1155 entries, 0 to 1154
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   START_DATE*  1155 non-null   str    
 1   END_DATE*    1155 non-null   str    
 2   CATEGORY*    1155 non-null   str    
 3   START*       1155 non-null   str    
 4   STOP*        1155 non-null   str    
 5   MILES*       1155 non-null   float64
 6   PURPOSE*     1155 non-null   str    
dtypes: float64(1), str(6)
memory usage: 63.3 KB


# Limpieza del dataset:

In [ ]:
ai_Buffer = ai_Buffer.clean_names(remove_special=True)
#usando IA mediante PyJanitor para limpiar nombres de columnas y eliminar caracteres especiales.

In [ ]:
df_base = ai_Buffer
df_base

,start_date,end_date,category,start,stop,miles,purpose
0,1/1/2016 21:11,1/1/2016 21:17,Business,Fort Pierce,Fort Pierce,5.1,Meal/Entertain
1,2/1/2016 1:25,2/1/2016 1:37,Business,Fort Pierce,Fort Pierce,5.0,No Especificado
2,2/1/2016 20:25,2/1/2016 20:38,Business,Fort Pierce,Fort Pierce,4.8,Errand/Supplies
3,5/1/2016 17:31,5/1/2016 17:45,Business,Fort Pierce,Fort Pierce,4.7,Meeting
4,6/1/2016 14:42,6/1/2016 15:49,Business,Fort Pierce,West Palm Beach,63.7,Customer Visit
...,...,...,...,...,...,...,...
1150,12/31/2016 1:07,12/31/2016 1:14,Business,Kar?chi,Kar?chi,0.7,Meeting
1151,12/31/2016 13:24,12/31/2016 13:42,Business,Kar?chi,Unknown Location,3.9,Temporary Site
1152,12/31/2016 15:03,12/31/2016 15:38,Business,Unknown Location,Unknown Location,16.2,Meeting
1153,12/31/2016 21:32,12/31/2016 21:50,Business,Katunayake,Gampaha,6.4,Temporary Site


## determinamos candidatos para indexar

In [50]:
df["PURPOSE*"].tail()


1150           Meeting
1151    Temporary Site
1152           Meeting
1153    Temporary Site
1154    Temporary Site
Name: PURPOSE*, dtype: str

In [51]:
df["PURPOSE*"].nunique()

11

## Purspose o "Proposito" es un buen candidato a indexar 


In [52]:
print(df["CATEGORY*"].unique())


<StringArray>
['Business', 'Personal']
Length: 2, dtype: str


In [53]:
df["CATEGORY*"].nunique()

2

In [54]:
counts_cat = df['CATEGORY*'].value_counts()
unique_cat = counts_cat.index
print("Recuento viajes por categoria:\n", counts_cat)
print("Unique values:", list(unique_cat))

Recuento viajes por categoria:
 CATEGORY*
Business    1078
Personal      77
Name: count, dtype: int64
Unique values: ['Business', 'Personal']


#  CATEGORY no es un buen candidato  indexar

### **dia del año** seria el candidato ideal a indexar, el formato actual de nuestro timestamp nos difficulta un poco su captura, pero podemos hacer un preprocesamiento para extraer el dia y luego indexar por ese campo.

In [65]:



indexador_fechas = recordlinkage.Index()
escaleno = StandardScaler()
indexador_proposito = recordlinkage.Index()
cubero = recordlinkage.Compare()


df_base['miles_scaled'] = escaleno.fit_transform(df_base[['miles']])

### indexamos por dia del año sin descartar los segundos:

In [ ]:
df_fechas = df_base.copy()

# Convert the string to a datetime object, then extract just the Date
# This safely strips the HH:mm so all rides on the same day are grouped
df_fechas['solo_fecha'] = pd.to_datetime(df_fechas['start_date']).dt.date

indexador_fechas.block('solo_fecha')
candidatos_fechas = indexador_fechas.index(df_fechas)